In [0]:
%run "/Workspace/Users/ceresrain@yahoo.com/Butterfly_effect/short-form-butterfly/config/00_config/config"

Config loaded.
Bronze: bootcamp_students.tiffani_ceresrain_bronze
Silver: bootcamp_students.tiffani_ceresrain_silver
Gold:   bootcamp_students.tiffani_ceresrain_gold
Films loaded: 50


In [0]:
# ============================================================
# BRONZE | 01_ingest_google_trends
# Pulls Google Trends search volume for each film
# in the 90-day window before release date
# ============================================================


# ── Install ─────────────────────────────────────────────────
%pip install pytrends

# ── TODO: Add ingestion logic in next session ────────────────
print("Bronze 01 shell ready.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 37.1 MB/s eta 0:00:00
Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.
Bronze 01 shell ready.


In [0]:
# — Ingest Google Trends for each film
from pytrends.request import TrendReq
import pandas as pd
import time
from datetime import datetime, timedelta

pytrends = TrendReq(hl='en-US', tz=360)

results = []

for film in FILM_LIST:
    title = film["title"]
    release_date = datetime.strptime(film["release_date"], "%Y-%m-%d")
    
    # 90-day window before release
    start = (release_date - timedelta(days=90)).strftime("%Y-%m-%d")
    end = release_date.strftime("%Y-%m-%d")
    timeframe = f"{start} {end}"
    
    try:
        pytrends.build_payload([title], timeframe=timeframe, geo='US')
        df = pytrends.interest_over_time()
        
        if not df.empty:
            df = df.reset_index()
            df["film"] = title
            df["release_date"] = film["release_date"]
            df["genre"] = film["genre"]
            df = df.rename(columns={title: "interest_score"})
            df = df[["date", "film", "release_date", "genre", "interest_score"]]
            results.append(df)
            print(f"✅ {title} — {len(df)} rows")
        else:
            print(f"⚠️ {title} — no data returned")
            
    except Exception as e:
        print(f"❌ {title} — error: {e}")
    
    time.sleep(15)  # avoid rate limiting

# Combine all results
if results:
    df_all = pd.concat(results, ignore_index=True)
    print(f"\nTotal rows: {len(df_all)}")
    display(df_all.head(20))
else:
    print("No data collected.")

✅ Batman Begins — 91 rows
✅ Brokeback Mountain — 91 rows
✅ Casino Royale — 91 rows
✅ The Departed — 91 rows
✅ No Country for Old Men — 91 rows
✅ There Will Be Blood — 91 rows
✅ Ratatouille — 91 rows
✅ The Dark Knight — 91 rows
✅ Inception — 91 rows
✅ Avatar — 91 rows
✅ The Social Network — 91 rows
✅ Interstellar — 91 rows
✅ Gone Girl — 91 rows
✅ The Wolf of Wall Street — 91 rows
✅ Gravity — 91 rows
✅ The Hangover — 91 rows
✅ Bridesmaids — 91 rows
✅ Paranormal Activity — 91 rows
✅ The Conjuring — 91 rows
✅ Mad Max: Fury Road — 91 rows
✅ The Revenant — 91 rows
✅ La La Land — 91 rows
✅ Get Out — 91 rows
✅ It — 91 rows
✅ Black Panther — 91 rows
✅ Crazy Rich Asians — 91 rows
✅ A Quiet Place — 91 rows
✅ Hereditary — 91 rows
✅ Bohemian Rhapsody — 91 rows
✅ Joker — 91 rows
✅ Parasite — 91 rows
✅ Tenet — 91 rows
✅ Dune — 91 rows
✅ The Batman — 91 rows
✅ Top Gun: Maverick — 91 rows
✅ Everything Everywhere All at Once — 91 rows
✅ Morbius — 91 rows
✅ Nope — 91 rows
✅ The Menu — 91 rows
✅ Oppenheim

date,film,release_date,genre,interest_score
2005-03-17T00:00:00.000Z,Batman Begins,2005-06-15,Action,3
2005-03-18T00:00:00.000Z,Batman Begins,2005-06-15,Action,5
2005-03-19T00:00:00.000Z,Batman Begins,2005-06-15,Action,3
2005-03-20T00:00:00.000Z,Batman Begins,2005-06-15,Action,6
2005-03-21T00:00:00.000Z,Batman Begins,2005-06-15,Action,3
2005-03-22T00:00:00.000Z,Batman Begins,2005-06-15,Action,4
2005-03-23T00:00:00.000Z,Batman Begins,2005-06-15,Action,5
2005-03-24T00:00:00.000Z,Batman Begins,2005-06-15,Action,4
2005-03-25T00:00:00.000Z,Batman Begins,2005-06-15,Action,5
2005-03-26T00:00:00.000Z,Batman Begins,2005-06-15,Action,7


In [0]:
# — Retry failed films only
failed_films = [
    {"title": "The Revenant",       "release_date": "2015-12-25", "genre": "Drama"},
    {"title": "Get Out",            "release_date": "2017-02-24", "genre": "Horror"},
    {"title": "Crazy Rich Asians",  "release_date": "2018-08-15", "genre": "Comedy"},
    {"title": "Hereditary",         "release_date": "2018-06-08", "genre": "Horror"},
]

retry_results = []

for film in failed_films:
    title = film["title"]
    release_date = datetime.strptime(film["release_date"], "%Y-%m-%d")
    start = (release_date - timedelta(days=90)).strftime("%Y-%m-%d")
    end = release_date.strftime("%Y-%m-%d")
    timeframe = f"{start} {end}"
    
    try:
        pytrends.build_payload([title], timeframe=timeframe, geo='US')
        df = pytrends.interest_over_time()
        if not df.empty:
            df = df.reset_index()
            df["film"] = title
            df["release_date"] = film["release_date"]
            df["genre"] = film["genre"]
            df = df.rename(columns={title: "interest_score"})
            df = df[["date", "film", "release_date", "genre", "interest_score"]]
            retry_results.append(df)
            print(f"✅ {title} — {len(df)} rows")
        else:
            print(f"⚠️ {title} — no data")
    except Exception as e:
        print(f"❌ {title} — {e}")
    
    time.sleep(20)

# Combine with original results and save
if retry_results:
    df_retry = pd.concat(retry_results, ignore_index=True)
    df_all = pd.concat([df_all, df_retry], ignore_index=True)
    print(f"\nUpdated total rows: {len(df_all)}")
else:
    print("Retries failed — try again later")

✅ The Revenant — 91 rows
✅ Get Out — 91 rows
✅ Crazy Rich Asians — 91 rows
✅ Hereditary — 91 rows

Updated total rows: 3731


In [0]:
# — Create schemas if they don't exist
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {BRONZE_PATH}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_PATH}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {GOLD_PATH}")

print(f"✅ Bronze schema ready: {BRONZE_PATH}")
print(f"✅ Silver schema ready: {SILVER_PATH}")
print(f"✅ Gold schema ready:   {GOLD_PATH}")

✅ Bronze schema ready: bootcamp_students.tiffani_ceresrain_bronze
✅ Silver schema ready: bootcamp_students.tiffani_ceresrain_silver
✅ Gold schema ready:   bootcamp_students.tiffani_ceresrain_gold


In [0]:
# — Save to Bronze Delta table
df_spark = spark.createDataFrame(df_all)

(df_spark
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{BRONZE_PATH}.google_trends_raw")
)

print(f"✅ Saved to {BRONZE_PATH}.google_trends_raw")
print(f"   Rows written: {df_spark.count()}")

# Quick verify
display(spark.table(f"{BRONZE_PATH}.google_trends_raw").limit(5))

✅ Saved to bootcamp_students.tiffani_ceresrain_bronze.google_trends_raw
   Rows written: 4550


date,film,release_date,genre,interest_score
2005-03-17T00:00:00.000Z,Batman Begins,2005-06-15,Action,3
2005-03-18T00:00:00.000Z,Batman Begins,2005-06-15,Action,5
2005-03-19T00:00:00.000Z,Batman Begins,2005-06-15,Action,3
2005-03-20T00:00:00.000Z,Batman Begins,2005-06-15,Action,6
2005-03-21T00:00:00.000Z,Batman Begins,2005-06-15,Action,3


In [0]:
# — Save updated Bronze table
df_spark = spark.createDataFrame(df_all)

(df_spark
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{BRONZE_PATH}.google_trends_raw")
)

print(f"✅ Saved to {BRONZE_PATH}.google_trends_raw")
print(f"   Rows written: {df_spark.count()}")

✅ Saved to bootcamp_students.tiffani_ceresrain_bronze.google_trends_raw
   Rows written: 1001
